# 13 Tabular Export

This notebook audits TorchLens tabular exports. `to_pandas`, `to_csv`, `to_parquet`, and `to_json` are available on the full trace, individual records, and the major accessors so users can analyze captures with ordinary DataFrame tooling.

The model has parameters, a buffer, multiple module records, and several layers so the DataFrame examples are meaningful but still small.

In [ ]:
from pathlib import Path
import sys

REPO_ROOT = Path.cwd()
if REPO_ROOT.name == "audit":
    REPO_ROOT = REPO_ROOT.parents[1]
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

In [ ]:
import json
from pathlib import Path

import pandas as pd
import torch
from torch import nn
import torchlens as tl

ARTIFACT_DIR = REPO_ROOT / "notebooks" / "audit"
ARTIFACT_DIR.mkdir(parents=True, exist_ok=True)

torch.manual_seed(59)


class TabularNet(nn.Module):
    """Small model with tabular metadata surfaces."""

    def __init__(self) -> None:
        """Initialize layers and a buffer."""

        super().__init__()
        self.encoder = nn.Linear(4, 5)
        self.decoder = nn.Linear(5, 2)
        self.register_buffer("offset", torch.tensor([0.25, -0.25]))

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        """Run the encoder, activation, decoder, and buffer add.

        Parameters
        ----------
        x:
            Input tensor with four features.

        Returns
        -------
        torch.Tensor
            Two-feature output tensor.
        """

        hidden = torch.relu(self.encoder(x))
        return self.decoder(hidden) + self.offset


model = TabularNet().eval()
x = torch.randn(3, 4)
trace = tl.trace(model, x, layers_to_save="all")
print(f"layers: {len(trace.layers)}")
print(f"params: {len(trace.params)}")
print(f"buffers: {len(trace.buffers)}")
print(f"modules: {len(trace.modules)}")

`Trace.to_pandas()` gives one row per visible layer with a broad set of analysis columns.

In [ ]:
layer_df = trace.to_pandas()
print(
    layer_df[["layer_label", "layer_type", "shape", "memory", "memory_str"]].to_string(index=False)
)

Once in pandas, ordinary sorting and filtering are enough for many audit questions.

In [ ]:
memory_rank = layer_df.sort_values("memory", ascending=False)[
    ["layer_label", "layer_type", "memory_str", "num_params"]
]
linear_only = layer_df[layer_df["layer_type"].str.contains("linear", case=False, na=False)][
    ["layer_label", "shape", "num_params"]
]
print("largest by memory:")
print(memory_rank.head(4).to_string(index=False))
print("\nlinear-like layers:")
print(linear_only.to_string(index=False))

Accessors export narrower tables for layers, modules, parameters, and buffers.

In [ ]:
surfaces = {
    "layers": trace.layers,
    "modules": trace.modules,
    "params": trace.params,
    "buffers": trace.buffers,
}
for name, surface in surfaces.items():
    frame = surface.to_pandas()
    print(
        f"{name:7s} rows={len(frame):2d} columns={len(frame.columns):2d} first_columns={list(frame.columns[:4])}"
    )

Individual records also have `to_pandas()`. This is useful when you want every field for one operation or one module call.

In [ ]:
first_linear = next(layer for layer in trace.layers if layer.func_name == "linear")
module_call = trace.module_calls["encoder:1"]
record_frames = {
    "layer_record": first_linear.to_pandas(),
    "module_call": module_call.to_pandas(),
}
for name, frame in record_frames.items():
    print(f"{name}: rows={len(frame)} columns={len(frame.columns)}")
    print(frame.iloc[:, :6].to_string(index=False))

The file exporters write the same table shapes. Parquet requires `pyarrow`, so this cell prints a note instead of failing when the optional dependency is missing.

In [ ]:
exports = {
    "trace": trace,
    "layers": trace.layers,
    "params": trace.params,
    "module_call": module_call,
}
for name, surface in exports.items():
    base = ARTIFACT_DIR / f"13_{name}"
    csv_path = base.with_suffix(".csv")
    json_path = base.with_suffix(".json")
    parquet_path = base.with_suffix(".parquet")

    surface.to_csv(csv_path)
    surface.to_json(json_path)
    try:
        surface.to_parquet(parquet_path)
        parquet_status = f"parquet rows={len(pd.read_parquet(parquet_path))}"
    except ImportError as exc:
        parquet_status = f"> NOTE: parquet skipped: {exc}"

    csv_rows = len(pd.read_csv(csv_path))
    json_rows = len(pd.read_json(json_path, orient="records"))
    print(f"{name:11s} csv_rows={csv_rows} json_rows={json_rows} {parquet_status}")

JSON files are regular records-oriented tables, so they are easy to inspect without pandas.

In [ ]:
json_path = ARTIFACT_DIR / "13_params.json"
records = json.loads(json_path.read_text())
print(f"json records: {len(records)}")
print({key: records[0][key] for key in ["address", "shape", "trainable"]})

> NOTE: `trace.ops` is the low-level op accessor in this build and does not expose the tabular quartet directly. Individual `Op`/`Layer` records and the user-facing accessors do.

## 🔧 Sandbox

Change the sort key or exported surface and re-run the table analysis.

In [ ]:
# TODO: try sandbox_sort_key = "num_params" or sandbox_surface = trace.modules.
sandbox_sort_key = "memory"
sandbox_surface = trace.to_pandas()
sandbox_columns = ["layer_label", "layer_type", sandbox_sort_key]
print(
    sandbox_surface.sort_values(sandbox_sort_key, ascending=False)[sandbox_columns]
    .head(5)
    .to_string(index=False)
)